In [1]:
import os

In [2]:
%pwd

'd:\\Project\\creditcard-fraud-detection-proj\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Project\\creditcard-fraud-detection-proj'

In [5]:
os.environ["MLFLOW_TRACKING_URI"] = (
    "https://dagshub.com/vinitvaidya/creditcard-fraud-detection-proj.mlflow"
)
os.environ["MLFLOW_TRACKING_USERNAME"] = "vinitvaidya"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "2508250030a01d4680c4d438d5a676b25d707b96"

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    classification_report_file_name: Path
    confusion_matrix: Path
    target_column: str
    mlflow_uri: str

In [7]:
from cred_card_proj.constants import *
from cred_card_proj.utils.common import read_yaml, create_directories, save_json

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.XGBoost
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            classification_report_file_name=config.classification_report_file_name,
            confusion_matrix=config.confusion_matrix,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/vinitvaidya/creditcard-fraud-detection-proj.mlflow",
        )

        return model_evaluation_config

In [9]:
import os
import pandas as pd
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

d:\Project\creditcard-fraud-detection-proj\fraud_detection_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred, pred_proba=None):
        accuracy = accuracy_score(actual, pred)
        precision = precision_score(actual, pred)
        recall = recall_score(actual, pred)
        f1 = f1_score(actual, pred)

        metrics = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
        }

        if pred_proba is not None:
            metrics["roc_auc"] = roc_auc_score(actual, pred_proba)

        return metrics

    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_fraud = model.predict(test_x)
            predicted_proba = (
                model.predict_proba(test_x)[:, 1]
                if hasattr(model, "predict_proba")
                else None
            )

            scores = self.eval_metrics(test_y, predicted_fraud, predicted_proba)

            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            for metric_name, metric_value in scores.items():
                mlflow.log_metric(metric_name, metric_value)

            # Full classification report as text artifact — useful for
            # per-class breakdown (fraud vs non-fraud) beyond the scalar metrics
            report_text = classification_report(test_y, predicted_fraud)
            report_path = Path(self.config.classification_report_file_name)
            report_path.write_text(report_text)
            mlflow.log_artifact(str(report_path))

            # Confusion matrix as artifact too — very useful for fraud detection
            # to see false negatives (missed fraud) vs false positives
            cm = confusion_matrix(test_y, predicted_fraud)
            cm_path = Path(self.config.confusion_matrix)
            save_json(path=cm_path, data={"confusion_matrix": cm.tolist()})
            mlflow.log_artifact(str(cm_path))

            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.xgboost.log_model(
                    model, "model", registered_model_name="XGBoostModel"
                )
            else:
                mlflow.xgboost.log_model(model, "model")

In [11]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-08-12 10:34:18,585: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-12 10:34:18,585: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-12 10:34:18,592: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-08-12 10:34:18,595: INFO: common: created directory at: artifacts]
[2026-08-12 10:34:18,595: INFO: common: created directory at: artifacts/model_evaluation]
[2026-08-12 10:34:20,388: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]
[2026-08-12 10:34:22,979: INFO: common: json file saved at: artifacts\model_evaluation\confusion_matrix.json]


2026/08/12 10:34:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'XGBoostModel' already exists. Creating a new version of this model...
2026/08/12 10:34:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoostModel, version 1
Created version '1' of model 'XGBoostModel'.


🏃 View run resilient-roo-174 at: https://dagshub.com/vinitvaidya/creditcard-fraud-detection-proj.mlflow/#/experiments/0/runs/02ac81cd3230491fbd4afd21c47218e4
🧪 View experiment at: https://dagshub.com/vinitvaidya/creditcard-fraud-detection-proj.mlflow/#/experiments/0
